In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import torch
import numpy as np
import os
import json
import random
from collections import Counter, defaultdict

FEATURES_DIR = "/kaggle/input/datasets/ptrnghieu/hi-ef-features-v2"

# Adjust if needed
if not os.path.exists(FEATURES_DIR):
    for root, dirs, files in os.walk("/kaggle/input"):
        if any(f.endswith('.pt') for f in files[:10]):
            FEATURES_DIR = root
            break
            
print(f"Features dir: {FEATURES_DIR}")

# Load index
with open(os.path.join(FEATURES_DIR, "mcis_index.json"), 'r') as f:
    mcis_index = json.load(f)

# Sample 500 random clips for analysis
all_files = [f for f in os.listdir(FEATURES_DIR) if f.endswith('.pt')]
sample_files = random.sample(all_files, min(500, len(all_files)))

print(f"Analyzing {len(sample_files)} / {len(all_files)} clips...\n")

# === Check 1: Feature statistics ===
face_norms, ori_norms, text_norms, audio_norms = [], [], [], []
face_detected_count = 0
audio_found_count = 0
nan_count = 0
zero_text_count = 0

for f in sample_files:
    data = torch.load(os.path.join(FEATURES_DIR, f), map_location='cpu', weights_only=False)
    
    face = data['face_features']
    ori = data['ori_features']
    text = data['text_feature']
    audio = data['audio_feature']
    
    # Check for NaN
    if torch.isnan(face).any() or torch.isnan(ori).any() or torch.isnan(audio).any():
        nan_count += 1
    
    face_norms.append(face.norm(dim=-1).mean().item())
    ori_norms.append(ori.norm(dim=-1).mean().item())
    text_norms.append(text.norm().item())
    audio_norms.append(audio.norm().item())
    
    if data.get('face_valid_mask'):
        face_detected_count += sum(data['face_valid_mask']) / len(data['face_valid_mask'])
    if data.get('audio_found', False):
        audio_found_count += 1
    if text.abs().sum() < 1e-6:
        zero_text_count += 1

n = len(sample_files)
print("=== Feature Statistics ===")
print(f"Face feature norms:  mean={np.mean(face_norms):.4f}, std={np.std(face_norms):.4f}, min={np.min(face_norms):.4f}, max={np.max(face_norms):.4f}")
print(f"Ori feature norms:   mean={np.mean(ori_norms):.4f}, std={np.std(ori_norms):.4f}, min={np.min(ori_norms):.4f}, max={np.max(ori_norms):.4f}")
print(f"Text feature norms:  mean={np.mean(text_norms):.4f}, std={np.std(text_norms):.4f}, min={np.min(text_norms):.4f}, max={np.max(text_norms):.4f}")
print(f"Audio feature norms: mean={np.mean(audio_norms):.4f}, std={np.std(audio_norms):.4f}, min={np.min(audio_norms):.4f}, max={np.max(audio_norms):.4f}")
print(f"\nNaN features: {nan_count}/{n}")
print(f"Zero text features: {zero_text_count}/{n}")
print(f"Face detection rate: {face_detected_count/n*100:.1f}% of frames")
print(f"Audio found: {audio_found_count}/{n} ({audio_found_count/n*100:.1f}%)")

# === Check 2: Do features separate emotions? ===
# Gather clip3+clip4 features grouped by emotion
emo_names = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
emo_features = defaultdict(list)

for entry in mcis_index:
    c4_emo = entry['clip4_emotion']
    if c4_emo < 0:
        continue
    c4_file = entry['feature_files'][3]
    path = os.path.join(FEATURES_DIR, c4_file)
    if os.path.exists(path):
        data = torch.load(path, map_location='cpu', weights_only=False)
        # Use mean face feature as a simple representation
        feat = data['face_features'].mean(dim=0)  # [512]
        emo_features[c4_emo].append(feat)

# Compute inter-class vs intra-class cosine similarity
print("\n=== Emotion Separability (face features, cosine similarity) ===")
centroids = {}
for emo_id in range(7):
    if emo_features[emo_id]:
        feats = torch.stack(emo_features[emo_id])
        centroids[emo_id] = feats.mean(dim=0)
        centroids[emo_id] = centroids[emo_id] / centroids[emo_id].norm()

print(f"\nInter-class cosine similarity (lower = better separation):")
header = f"{'':>10}" + "".join(f"{emo_names[i][:7]:>8}" for i in range(7))
print(header)
for i in range(7):
    if i not in centroids:
        continue
    row = f"{emo_names[i]:>10}"
    for j in range(7):
        if j not in centroids:
            row += f"{'N/A':>8}"
        else:
            sim = torch.dot(centroids[i], centroids[j]).item()
            row += f"{sim:>8.3f}"
    row += f"  (n={len(emo_features[i])})"
    print(row)

# Intra-class variance
print(f"\nIntra-class cosine similarity to centroid (higher = tighter clusters):")
for emo_id in range(7):
    if emo_id in centroids and emo_features[emo_id]:
        feats = torch.stack(emo_features[emo_id])
        feats = feats / feats.norm(dim=-1, keepdim=True)
        sims = (feats @ centroids[emo_id]).mean().item()
        print(f"  {emo_names[emo_id]:>10}: {sims:.3f} (n={len(emo_features[emo_id])})")

Features dir: /kaggle/input/datasets/ptrnghieu/hi-ef-features-v2
Analyzing 500 / 7925 clips...

=== Feature Statistics ===
Face feature norms:  mean=1.0000, std=0.0000, min=1.0000, max=1.0000
Ori feature norms:   mean=1.0000, std=0.0000, min=1.0000, max=1.0000
Text feature norms:  mean=0.9980, std=0.0447, min=0.0000, max=1.0000
Audio feature norms: mean=141.5288, std=13.3903, min=101.0136, max=194.7122

NaN features: 0/500
Zero text features: 1/500
Face detection rate: 95.1% of frames
Audio found: 500/500 (100.0%)

=== Emotion Separability (face features, cosine similarity) ===

Inter-class cosine similarity (lower = better separation):
             angry disgust    fear   happy neutral     sad surpris
     angry   1.000   0.999   0.989   0.992   0.997   0.995   0.994  (n=525)
   disgust   0.999   1.000   0.989   0.991   0.997   0.993   0.994  (n=260)
      fear   0.989   0.989   1.000   0.991   0.992   0.993   0.994  (n=38)
     happy   0.992   0.991   0.991   1.000   0.996   0.996   

In [3]:
# Check audio feature diversity
audio_feats = []
for f in sample_files[:200]:
    data = torch.load(os.path.join(FEATURES_DIR, f), map_location='cpu', weights_only=False)
    audio_feats.append(data['audio_feature'])

audio_feats = torch.stack(audio_feats)
# Normalize for cosine similarity
audio_normed = audio_feats / audio_feats.norm(dim=-1, keepdim=True)
# Pairwise cosine similarity
cos_sim = (audio_normed @ audio_normed.T)
# Exclude diagonal
mask = ~torch.eye(200, dtype=bool)
print(f"Audio pairwise cosine similarity:")
print(f"  Mean: {cos_sim[mask].mean():.4f}")
print(f"  Std:  {cos_sim[mask].std():.4f}")
print(f"  Min:  {cos_sim[mask].min():.4f}")
print(f"  Max:  {cos_sim[mask].max():.4f}")

# Compare: same check for face features
face_feats = []
for f in sample_files[:200]:
    data = torch.load(os.path.join(FEATURES_DIR, f), map_location='cpu', weights_only=False)
    face_feats.append(data['face_features'].mean(dim=0))

face_feats = torch.stack(face_feats)
face_normed = face_feats / face_feats.norm(dim=-1, keepdim=True)
cos_sim_face = (face_normed @ face_normed.T)
print(f"\nFace pairwise cosine similarity:")
print(f"  Mean: {cos_sim_face[mask].mean():.4f}")
print(f"  Std:  {cos_sim_face[mask].std():.4f}")
print(f"  Min:  {cos_sim_face[mask].min():.4f}")
print(f"  Max:  {cos_sim_face[mask].max():.4f}")

Audio pairwise cosine similarity:
  Mean: 0.9878
  Std:  0.0080
  Min:  0.9372
  Max:  0.9996

Face pairwise cosine similarity:
  Mean: 0.7999
  Std:  0.0664
  Min:  0.4854
  Max:  0.9910
